# PydanticAI together with Pydantic Models

In [1]:
from model import nvidia_model
from pydantic_ai import Agent

employee_simulator = Agent(model=nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b"),
instructions="Du är expert på att simulera anställda i Sverige. Simulera i detta formatet"
"""
    name: "Bertil",
    age: 40,
    salary: 35000,
    role: "IT-supporter
    ort: Stockholm
""")

result = await employee_simulator.run("Ge mig en data engineer ")
print(result.output)

name: "Erik",
age: 34,
salary: 56000,
role: "Data Engineer",
ort: Stockholm


## Force a structured output using pydantic

In [6]:
from pydantic import BaseModel, Field

class EmployeeModel(BaseModel):
    name: str = Field(
        description="för och efternamn, inte jättelångt, svenska eller internationella namn"
    )
    age: int = Field(description="ålder från 18 till 67 år", ge=18, le=67)
    salary: int = Field(
        description="lön mellan 30k-60k, ju mer senior position, desto högre lön",
        ge=30_000,
        lt=60_000,
    )
    role: str = Field(description="yrke/roll")
    description: str = "kort beskrivning, 2 meningar på rollen"

employee1 = EmployeeModel(name="Bertil", age=40, salary= 50_000, role="It-supporter", description="Hjälper folk med IT")

In [9]:
employee_simulator = Agent(model=nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b"),
instructions="Du är expert på att simulera anställda i Sverige",
)

result = await employee_simulator.run("Ge mig en data engineer i Stockholm", output_type=EmployeeModel)

result

AgentRunResult(output=EmployeeModel(name='Erik Johansson', age=32, salary=48000, role='Data Engineer', description='Erik arbetar som Data Engineer i Stockholm och ansvarar för att bygga och underhålla datapipelines samt molnbaserade datalösningar. Han har erfarenhet av Python, SQL och plattformar som Azure och GCP.'))

In [16]:
employee2 = result.output

In [18]:
employee2.name + " är cool"

'Erik Johansson är cool'

In [20]:
employee2.__dict__.keys()

dict_keys(['name', 'age', 'salary', 'role', 'description'])

In [21]:
employee2.__dict__.values()

dict_values(['Erik Johansson', 32, 48000, 'Data Engineer', 'Erik arbetar som Data Engineer i Stockholm och ansvarar för att bygga och underhålla datapipelines samt molnbaserade datalösningar. Han har erfarenhet av Python, SQL och plattformar som Azure och GCP.'])

In [22]:
for key, value in employee2.__dict__.items():
    print(f"{key}: {value}")

name: Erik Johansson
age: 32
salary: 48000
role: Data Engineer
description: Erik arbetar som Data Engineer i Stockholm och ansvarar för att bygga och underhålla datapipelines samt molnbaserade datalösningar. Han har erfarenhet av Python, SQL och plattformar som Azure och GCP.


# Simulate many employees

In [24]:
result = await employee_simulator.run("Ge mig totalt tio anställda inom IT/Data/Sälj",
                                      output_type=list[EmployeeModel])

result

AgentRunResult(output=[EmployeeModel(name='Erik Andersson', age=34, salary=48000, role='Systemutvecklare', description='Erfaren backend-utvecklare med fokus på Java och molnlösningar. Arbetar med att bygga skalbara API:er för företagets kärnsystem.'), EmployeeModel(name='Maria Johansson', age=29, salary=42000, role='Dataanalytiker', description='Ansvarar för att omvandla rådata till affärsinsikter med hjälp av SQL och Python. Stödjer säljavdelningen med kundanalyser och rapporter.'), EmployeeModel(name='Lars Karlsson', age=45, salary=55000, role='IT-säkerhetsspecialist', description='Säkerställer att företagets nätverk och system skyddas mot intrång och cyberhot. Genomför regelbundna riskbedömningar och säkerhetsutbildningar.'), EmployeeModel(name='Sofia Nilsson', age=26, salary=32000, role='IT-supporttekniker', description='Första linjens support för anställda med problem kring datorer, mobiltelefoner och programvara. Hanterar ärenden i ticket-systemet med fokus på snabb lösning.'), E

In [41]:
employees = result.output

salary = 0
for employee in employees:
    salary = employee.salary

salary/len(employees)


5300.0

# Convert to dataframe

In [42]:
employees[0].model_dump()

{'name': 'Erik Andersson',
 'age': 34,
 'salary': 48000,
 'role': 'Systemutvecklare',
 'description': 'Erfaren backend-utvecklare med fokus på Java och molnlösningar. Arbetar med att bygga skalbara API:er för företagets kärnsystem.'}

In [47]:
import pandas as pd
df = pd.DataFrame([employee.model_dump() for employee in employees])
df.head()

,name,age,salary,role,description
0,Erik Andersson,34,48000,Systemutvecklare,Erfaren backend-utvecklare med fokus på Java o...
1,Maria Johansson,29,42000,Dataanalytiker,Ansvarar för att omvandla rådata till affärsin...
2,Lars Karlsson,45,55000,IT-säkerhetsspecialist,Säkerställer att företagets nätverk och system...
3,Sofia Nilsson,26,32000,IT-supporttekniker,Första linjens support för anställda med probl...
4,David Petersson,38,51000,Databasadministratör,Förvaltar och optimerar företagets SQL-databas...


In [ ]:
df.to_csv("employees.csv", index=None)

_IncompleteInputError: incomplete input (1653361438.py, line 1)